<a href="https://colab.research.google.com/github/ruchit0002/Edgesense/blob/main/Edgesense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import time

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sst_train.csv to sst_train.csv


In [ ]:
file = list(uploaded.keys())[0]

df = pd.read_csv(file)
df.columns = df.columns.str.strip()

print("Columns:", df.columns)
df.head()

Columns: Index(['label', 'cleaned_text'], dtype='object')


,label,cleaned_text
0,3,The Rock is destined to be th...
1,4,The gorgeously elaborate continuation ...
2,3,Singer\/composer Bryan Adams contri...
3,2,You 'd think by now America woul...
4,3,Yet the act is still charming here


In [ ]:
text_col = None
label_col = None

for col in df.columns:
    sample = str(df[col].iloc[0])

    if isinstance(sample, str) and len(sample) > 20:
        text_col = col
    else:
        label_col = col

if text_col is None:
    text_col = df.columns[0]

if label_col is None:
    label_col = df.columns[1]

print("Text column:", text_col)
print("Label column:", label_col)

Text column: cleaned_text
Label column: label


In [ ]:
def convert_label(x):
    try:
        return int(x)
    except:
        x = str(x).lower()
        if "pos" in x:
            return 1
        elif "neg" in x:
            return 0
        else:
            return None

df[label_col] = df[label_col].apply(convert_label)

df = df.dropna(subset=[label_col])
df[label_col] = df[label_col].astype(int)

In [ ]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.eval()

print("Model loaded.")

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded.


In [ ]:
correct = 0
total = 0
total_time = 0

for i in range(min(100, len(df))):

    text = str(df[text_col].iloc[i])
    label = int(df[label_col].iloc[i])

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    start = time.time()

    with torch.no_grad():
        outputs = model(**inputs)

    end = time.time()

    total_time += (end - start)

    pred = torch.argmax(outputs.logits).item()

    if pred == label:
        correct += 1

    total += 1

accuracy = correct / total

print("=== Results ===")
print("Samples tested:", total)
print("Accuracy:", round(accuracy*100,2), "%")
print("Average Inference Time:", round((total_time/total)*1000,2), "ms")

=== Results ===
Samples tested: 100
Accuracy: 1.0 %
Average Inference Time: 106.98 ms


In [ ]:
while True:
    user_input = input("Enter text (or type exit): ")

    if user_input.lower() == "exit":
        break

    inputs = tokenizer(user_input, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits).item()

    result = "Positive" if pred == 1 else "Negative"

    print("Prediction:", result)

Enter text (or type exit): The movie was okay but a bit boring I liked the design but performance is poor
Prediction: Negative
Enter text (or type exit): Yeah great, just what I needed 
Prediction: Positive
Enter text (or type exit): exit
